# 🧠 ResNet Cookbook: Understanding Residual Learning & Skip Connections

**Experiment 5 — Deep Learning Lab**

---

## 📋 Learning Objectives

By the end of this cookbook, you will be able to:

1. **Explain** the vanishing gradient problem and why it occurs in deep networks
2. **Derive** the residual learning formulation $y = \mathcal{F}(x) + x$ and its gradient properties
3. **Compare** four network configurations and interpret their training dynamics
4. **Prove** — with per-layer evidence — that skip connections fix gradient flow

## 🔬 Experiment Overview

We train **four** model variants on the **SVHN** (Street View House Numbers) dataset:

| # | Configuration | Residual Blocks? | Skip (`F(x)+x`)? |
|---|--------------|:-:|:-:|
| 1 | Simple CNN | ❌ | ❌ |
| 2 | Simple CNN + Skip | ❌ | ✅ |
| 3 | ResNet (no skip) | ✅ | ❌ |
| 4 | ResNet (with skip) | ✅ | ✅ |

All models share the same depth (~20 layers), dataset, hyperparameters, and initialization.

---
# Part I: Theory
---

## 1. The Vanishing Gradient Problem

### Why Deep Networks Struggle

Neural networks learn by **backpropagation**. For a network with $L$ layers, the gradient at layer $l$ is:

$$\frac{\partial \mathcal{L}}{\partial W_l} = \frac{\partial \mathcal{L}}{\partial a_L} \cdot \prod_{i=l}^{L-1} \frac{\partial a_{i+1}}{\partial a_i}$$

### The Core Issue

Each Jacobian factor $\frac{\partial a_{i+1}}{\partial a_i}$ depends on the activation function and weights:

- **Sigmoid/Tanh**: Derivatives bounded in $(0, 0.25]$ / $(0, 1]$
- **ReLU**: 0 or 1, but weight matrices can still cause shrinkage

When many factors < 1, the product **exponentially decays**:

$$\prod_{i=l}^{L-1} \frac{\partial a_{i+1}}{\partial a_i} \approx \alpha^{L-l}, \quad |\alpha| < 1$$

**Result**: Early layers receive tiny gradients → tiny weight updates → they stop learning.

### Visual Intuition

Think of it like a game of telephone — the error signal starts strong at the output but gets **weakened** as it passes backward through each layer. By the time it reaches the first layers, it's barely a whisper.

> **Key Insight**: The deeper the network, the worse this problem gets.

## 2. The Residual Learning Framework (He et al., 2015)

### The Big Idea

Instead of learning the desired mapping $\mathcal{H}(x)$ directly, learn the **residual**:

$$\mathcal{F}(x) = \mathcal{H}(x) - x \quad \Rightarrow \quad y = \mathcal{F}(x) + x$$

- $\mathcal{F}(x)$ = what the layers learn (the **residual**)
- $x$ = the **identity shortcut** (skip connection)

### Why Learning Residuals is Easier

If the optimal mapping is close to identity:
- **Without skip**: Must learn $\mathcal{H}(x) \approx x$ — complex through nonlinear layers
- **With skip**: Only learn $\mathcal{F}(x) \approx 0$ — pushing weights toward zero is trivial!

### Residual Block Structure

```
Input (x)
  │
  ├──────────────────┐
  │                  │ (skip / identity)
  ▼                  │
Conv → BN → ReLU    │
  │                  │
  ▼                  │
Conv → BN            │
  │                  │
  ▼                  │
  + ◄────────────────┘  ← Element-wise addition
  │
  ▼
ReLU → Output (y = F(x) + x)
```

When dimensions differ, a **1×1 conv projection shortcut** is used: $y = \mathcal{F}(x) + W_s \cdot x$

## 3. Why Skip Connections Fix Gradient Flow

### The Mathematical Proof

For $y = \mathcal{F}(x) + x$, the backward gradient is:

$$\frac{\partial y}{\partial x} = \frac{\partial \mathcal{F}(x)}{\partial x} + \mathbf{I}$$

### Why This Changes Everything

**Plain network** chain rule:
$$\frac{\partial \mathcal{L}}{\partial x_l} = \frac{\partial \mathcal{L}}{\partial x_L} \cdot \prod_{i=l}^{L-1} \frac{\partial \mathcal{F}_i}{\partial x_i} \quad \text{→ can vanish}$$

**Residual network** chain rule:
$$\frac{\partial \mathcal{L}}{\partial x_l} = \frac{\partial \mathcal{L}}{\partial x_L} \cdot \prod_{i=l}^{L-1} \left(\frac{\partial \mathcal{F}_i}{\partial x_i} + \mathbf{I}\right) \quad \text{→ stays healthy}$$

The $+\mathbf{I}$ ensures **even if $\frac{\partial \mathcal{F}_i}{\partial x_i}$ is small, the gradient still flows.**

### Comparison Table

| Aspect | Plain Network | Residual Network |
|--------|:-:|:-:|
| Gradient path | Single road through layers | Highway + local roads |
| If one layer blocks | All upstream layers starve | Highway keeps flowing |
| Early layer updates | Exponentially weaker | Maintained by identity path |
| Gradient magnitude | $O(\alpha^L)$, can vanish | $O(1)$, stays healthy |

> **Key Takeaway**: Skip connections provide a **mathematical guarantee** that gradients can flow to any layer.

---
# Part II: Experiments
---

## 4. Experiment Setup

### Dataset: SVHN (Street View House Numbers)
- **Task**: Classify 32×32 RGB images of digits (0–9)
- **Training**: 73,257 images | **Test**: 26,032 images

### Common Hyperparameters

| Parameter | Value |
|-----------|-------|
| Batch Size | 128 |
| Epochs | 20 |
| Optimizer | Adam (lr=1e-3) |
| Scheduler | StepLR (step=10, γ=0.5) |
| Loss | CrossEntropyLoss |
| Init | Kaiming Normal |

### Metrics Tracked Per Conv Layer

| Metric | Tells Us |
|--------|----------|
| **Gradient Norm (L2)** | How strong is the learning signal? |
| **Weight Norm (L2)** | Is capacity being utilized? |
| **Weight Delta (L2)** | How much did weights change this epoch? |
| **Gradient Ratio** | First/last layer grad — is flow balanced? |

> Low gradient norm + low weight delta = layer is **not learning** = vanishing gradients.

---
## 5. Case 1: Simple CNN — No Residual, No Skip

### Architecture
Plain stack of `Conv → BN → ReLU`. No residual structure, no shortcuts.

```
Input (3×32×32) → Conv stem (64ch)
→ 3× Conv-BN-ReLU (64ch, 32×32)
→ 3× Conv-BN-ReLU (128ch, 16×16)
→ 3× Conv-BN-ReLU (256ch, 8×8)
→ AvgPool → FC(256→10)
```

### Expected: Early layers show weaker gradients and updates than later layers.

In [ ]:
# Simple CNN Model (No Residual, No Skip)
import torch.nn as nn

class ConvBnRelu(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))
    def forward(self, x): return self.block(x)

class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.stem = ConvBnRelu(3, 64)
        channels = [64,64,64, 128,128,128, 256,256,256]
        layers, in_ch = [], 64
        for idx, out_ch in enumerate(channels):
            stride = 2 if idx in (3, 6) else 1
            layers.append(ConvBnRelu(in_ch, out_ch, stride=stride))
            in_ch = out_ch
        self.features = nn.Sequential(*layers)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(256, num_classes)
    def forward(self, x):
        x = self.stem(x); x = self.features(x)
        return self.fc(torch.flatten(self.avgpool(x), 1))

### Results: Simple CNN


**📈 Accuracy Curves**
![](../wo_resnet_wo_skip/plots/accuracy_curves.png)


**📉 Loss Curves**
![](../wo_resnet_wo_skip/plots/loss_curves.png)


**🔬 Epoch Gradient Norms**
![](../wo_resnet_wo_skip/plots/epoch_gradient_norms.png)


**🌡️ Gradient Heatmap**
![](../wo_resnet_wo_skip/plots/gradient_heatmap.png)


**📊 Gradient Ratio**
![](../wo_resnet_wo_skip/plots/gradient_ratio.png)


**⚖️ Weight Norms**
![](../wo_resnet_wo_skip/plots/weight_norms.png)


**📐 Weight Deltas**
![](../wo_resnet_wo_skip/plots/weight_deltas.png)


**🎯 Confusion Matrix**
![](../wo_resnet_wo_skip/plots/confusion_matrix.png)


### 🔍 Observations — Simple CNN

1. **Gradient Vanishing**: Early layer gradient norms are noticeably **lower** than later layers
2. **Weak Early Updates**: Weight deltas decrease for earlier layers
3. **Gradient Ratio < 1**: First/last gradient ratio is well below 1.0
4. **Baseline Reference**: Performance is limited by poor gradient flow to early feature extractors

---
## 6. Case 2: Simple CNN with Skip Connections

Same plain CNN structure as Case 1, but with **skip connections** added.
Tests whether skip connections alone (without residual blocks) improve gradient flow.

### Expected: Early layer gradients stronger than Case 1.

### Results: CNN + Skip


**📈 Accuracy**
![](../wo_resnet_w_skip/plots/accuracy_curves.png)


**📉 Loss**
![](../wo_resnet_w_skip/plots/loss_curves.png)


**🔬 Epoch Gradients**
![](../wo_resnet_w_skip/plots/epoch_gradient_norms.png)


**🌡️ Gradient Heatmap**
![](../wo_resnet_w_skip/plots/gradient_heatmap.png)


**📊 Gradient Ratio**
![](../wo_resnet_w_skip/plots/gradient_ratio.png)


**⚖️ Weight Norms**
![](../wo_resnet_w_skip/plots/weight_norms.png)


**📐 Weight Deltas**
![](../wo_resnet_w_skip/plots/weight_deltas.png)


**🎯 Confusion Matrix**
![](../wo_resnet_w_skip/plots/confusion_matrix.png)


**🖼️ Sample Predictions**
![](../wo_resnet_w_skip/plots/sample_predictions.png)


### 🔍 Observations — CNN + Skip

1. **Improved Gradient Flow**: Skip connections visibly improve early layer gradient norms
2. **Better Ratio**: First/last gradient ratio moves closer to 1.0
3. **Stronger Updates**: Early layers now get more meaningful weight updates

> Skip connections **alone** are beneficial, even without residual block design.

---
## 7. Case 3: ResNet — WITHOUT Skip Connections

ResNet-20 architecture with `PlainBlock` (two convs per block) but **no `F(x)+x` addition**.
Isolates the question: *Is block structure alone sufficient?*

```python
def forward(self, x):  # PlainBlock
    out = self.relu(self.bn1(self.conv1(x)))
    out = self.relu(self.bn2(self.conv2(out)))  # NO + x
    return out
```

In [ ]:
# PlainBlock: ResNet shape WITHOUT skip connection
class PlainBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        # NO skip connection
    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.relu(self.bn2(self.conv2(out)))  # ← No + x!
        return out

### Results: ResNet without Skip


**📈 Accuracy**
![](../outputs_resnet_wo_skip/plots/accuracy_curves.png)


**📉 Loss**
![](../outputs_resnet_wo_skip/plots/loss_curves.png)


**🔬 Epoch Gradients**
![](../outputs_resnet_wo_skip/plots/epoch_gradient_norms.png)


**🌡️ Gradient Heatmap**
![](../outputs_resnet_wo_skip/plots/gradient_heatmap.png)


**📊 Gradient Ratio**
![](../outputs_resnet_wo_skip/plots/gradient_ratio.png)


**⚖️ Weight Norms**
![](../outputs_resnet_wo_skip/plots/weight_norms.png)


**📐 Weight Deltas**
![](../outputs_resnet_wo_skip/plots/weight_deltas.png)


**🔢 Batch Gradient Norms**
![](../outputs_resnet_wo_skip/plots/batch_gradient_norms.png)


**🎯 Confusion Matrix**
![](../outputs_resnet_wo_skip/plots/confusion_matrix.png)


**🖼️ Predictions**
![](../outputs_resnet_wo_skip/plots/sample_predictions.png)


**🖼️ SVHN Samples**
![](../outputs_resnet_wo_skip/plots/sample_images.png)


**📋 Architecture**
![](../outputs_resnet_wo_skip/tables/table1_architecture.png)


**📋 Performance**
![](../outputs_resnet_wo_skip/tables/table2_performance.png)


**📋 Training Metrics**
![](../outputs_resnet_wo_skip/tables/table3_training_metrics.png)


**📋 Gradient Summary**
![](../outputs_resnet_wo_skip/tables/table4_gradient_summary.png)


### 🔍 Observations — ResNet (no skip)

1. **Gradient Vanishing Persists**: Removing skip addition causes same vanishing pattern
2. **Block Structure ≠ Solution**: Two-conv-per-block alone does NOT fix gradient flow
3. **Early Layer Starvation**: Gradient heatmap shows clear weakening in early layers

> **Critical**: Removing just `+ x` from ResNet blocks degrades gradient flow dramatically.

---
## 8. Case 4: ResNet WITH Skip Connections ✅

Complete ResNet-20 with **skip connections**: $y = \mathcal{F}(x) + x$

```python
def forward(self, x):  # BasicBlock
    identity = self.skip(x)                   # Skip path
    out = self.relu(self.bn1(self.conv1(x)))
    out = self.bn2(self.conv2(out))
    out += identity                            # ✅ F(x) + x
    return self.relu(out)
```

**Architecture**: Initial Conv + 3 stages × 3 BasicBlocks + AvgPool + FC = **20 layers**

In [ ]:
# BasicBlock: ResNet WITH skip connection
class BasicBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.skip = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.skip = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch))
    def forward(self, x):
        identity = self.skip(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += identity  # ✅ SKIP CONNECTION
        return self.relu(out)

### Results: ResNet WITH Skip


**📈 Accuracy**
![](../outputs_resnet_with_skip/plots/accuracy_curves.png)


**📉 Loss**
![](../outputs_resnet_with_skip/plots/loss_curves.png)


**🔬 Epoch Gradients**
![](../outputs_resnet_with_skip/plots/epoch_gradient_norms.png)


**🌡️ Gradient Heatmap**
![](../outputs_resnet_with_skip/plots/gradient_heatmap.png)


**📊 Gradient Ratio**
![](../outputs_resnet_with_skip/plots/gradient_ratio.png)


**⚖️ Weight Norms**
![](../outputs_resnet_with_skip/plots/weight_norms.png)


**📐 Weight Deltas**
![](../outputs_resnet_with_skip/plots/weight_deltas.png)


**🔢 Batch Gradients**
![](../outputs_resnet_with_skip/plots/batch_gradient_norms.png)


**🎯 Confusion Matrix**
![](../outputs_resnet_with_skip/plots/confusion_matrix.png)


**🖼️ Predictions**
![](../outputs_resnet_with_skip/plots/sample_predictions.png)


**🖼️ SVHN Samples**
![](../outputs_resnet_with_skip/plots/sample_images.png)


### 🔍 Observations — ResNet with Skip ✅

1. **Healthy Gradient Flow**: Gradient norms are **consistent across all layers**
2. **Uniform Updates**: All layers receive meaningful weight updates, including earliest ones
3. **Gradient Ratio ≈ 1**: First/last gradient ratio stays close to 1.0
4. **Best Accuracy**: Highest test accuracy among all four configurations
5. **Uniform Heatmap**: No stark contrast between early and late layers

> **Definitive proof**: `+ x` eliminates vanishing gradients and enables effective training.

---
# Part III: Comparative Analysis
---

## 9. Summary Comparison

| Metric | Simple CNN | CNN+Skip | ResNet(no skip) | ResNet(skip) |
|--------|:-:|:-:|:-:|:-:|
| Early layer gradient | Weak ⚠️ | Improved ✅ | Weak ⚠️ | Strong ✅ |
| Gradient ratio | ≪ 1 ❌ | ~1 | ≪ 1 ❌ | ≈ 1 ✅ |
| Early weight updates | Small | Moderate | Small | Large ✅ |
| Vanishing gradients? | Yes ❌ | Reduced | Yes ❌ | No ✅ |
| Relative accuracy | Baseline | Better | ~Baseline | Best ✅ |

### What This Proves

1. **Depth alone causes problems** (Case 1)
2. **Skip connections help any architecture** (Case 2)
3. **Block design alone is insufficient** (Case 3)
4. **Residual learning is the complete solution** (Case 4)

---
## 10. Key Takeaways & Conclusion

### Core Message

**ResNet's innovation is the skip connection**, not block structure. $y = F(x) + x$ creates a gradient highway that:
1. Prevents gradient vanishing via the identity Jacobian term
2. Makes learning identity mappings trivial ($F(x) \to 0$)
3. Enables 100+ layer networks (ResNet-152, etc.)

### Quick Reference

| Concept | Formula |
|---------|--------|
| Plain mapping | $y = \mathcal{F}(x)$ |
| Residual mapping | $y = \mathcal{F}(x) + x$ |
| Plain gradient | $\frac{\partial y}{\partial x} = \frac{\partial \mathcal{F}}{\partial x}$ |
| Residual gradient | $\frac{\partial y}{\partial x} = \frac{\partial \mathcal{F}}{\partial x} + \mathbf{I}$ |

### Viva Checklist ✅

1. Start with vanishing gradient chain-rule math
2. Define $y=F(x)$ vs $y=F(x)+x$
3. Explain why $+\mathbf{I}$ creates gradient highway
4. Show per-layer gradient norms, heatmaps, ratios
5. Conclude with first-layer vs last-layer evidence

> 💡 Strongest argument = **per-layer training dynamics**, not just top-line accuracy.

### References
- He et al. (2015). *Deep Residual Learning for Image Recognition*. CVPR 2016.
- SVHN: Netzer et al. *Reading Digits in Natural Images*. NIPS Workshop 2011.

---
# Part IV: Vanishing Gradient Comparison — Live Proof 🔬
---

## 11. Self-Contained Experiment: Train, Compare, Prove

This section is **fully self-contained** and can run directly on **Google Colab**.
We train all 4 model variants from scratch, track **per-layer gradients** during backpropagation,
and generate comparison graphs and tables to **definitively prove** that skip connections
eliminate the vanishing gradient problem.

### What We Will Prove

| Claim | Evidence |
|-------|----------|
| Without skip: gradients vanish in early layers | Per-layer gradient norm bar chart |
| Without skip: weights can't update | Per-layer weight delta comparison |
| Identity mapping alone fails | $y = \\mathcal{F}(x)$ gradient decay vs $y = \\mathcal{F}(x) + x$ |
| Skip connections fix gradient flow | Gradient ratio ≈ 1.0 for skip models |
| ResNet + Skip = best overall | Accuracy/Loss comparison |

In [ ]:
# ============================================================
# PART IV — SETUP: Install dependencies & detect hardware
# ============================================================
# Uncomment the line below if running on Google Colab:
# !pip install torch torchvision matplotlib numpy -q

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import os, time
from collections import defaultdict

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'PyTorch: {torch.__version__}')

# Hyperparameters
BATCH_SIZE = 128
EPOCHS = 10  # Colab-friendly (10 epochs × 4 models ≈ 20-30 min on T4)
LR = 1e-3
NUM_CLASSES = 10

# Data
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize([0.4377, 0.4438, 0.4728], [0.1980, 0.2010, 0.1970])])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.4377, 0.4438, 0.4728], [0.1980, 0.2010, 0.1970])])

train_ds = datasets.SVHN(root='./svhn_data', split='train', download=True, transform=transform_train)
test_ds  = datasets.SVHN(root='./svhn_data', split='test',  download=True, transform=transform_test)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f'Train: {len(train_ds)} | Test: {len(test_ds)}')

### 11.1 All Four Model Definitions

We define all four architectures with the **same depth (~20 layers)**, channels, and initialization:

| Model | Block Type | Skip (`F(x)+x`)? | Key Difference |
|-------|-----------|:-:|----------------|
| `SimpleCNN` | Conv-BN-ReLU stack | ❌ | Plain sequential layers |
| `SimpleCNNSkip` | Conv-BN-ReLU + shortcut | ✅ | Skip added to plain CNN |
| `ResNetNoSkip` | Two-conv blocks (PlainBlock) | ❌ | ResNet structure, no `+x` |
| `ResNetWithSkip` | Two-conv blocks (BasicBlock) | ✅ | Full residual: `F(x)+x` |

In [ ]:
# ============================================================
# MODEL 1: Simple CNN (No Residual, No Skip)
# ============================================================
class ConvBnRelu(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU(inplace=True)
    def forward(self, x):
        return self.relu(self.bn(self.conv(x)))

class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.stem = ConvBnRelu(3, 64)
        channels = [64,64,64, 128,128,128, 256,256,256]
        layers, in_ch = [], 64
        for idx, out_ch in enumerate(channels):
            stride = 2 if idx in (3, 6) else 1
            layers.append(ConvBnRelu(in_ch, out_ch, stride=stride))
            in_ch = out_ch
        self.features = nn.Sequential(*layers)
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(256, num_classes)
        self._init_weights()
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d): nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d): nn.init.constant_(m.weight, 1); nn.init.constant_(m.bias, 0)
    def forward(self, x):
        x = self.stem(x); x = self.features(x)
        return self.fc(torch.flatten(self.avgpool(x), 1))

# ============================================================
# MODEL 2: Simple CNN WITH Skip Connections
# ============================================================
class ConvBnReluSkip(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU(inplace=True)
        self.skip = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.skip = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch))
    def forward(self, x):
        return self.relu(self.bn(self.conv(x)) + self.skip(x))  # ✅ Skip!

class SimpleCNNSkip(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.stem = ConvBnRelu(3, 64)
        channels = [64,64,64, 128,128,128, 256,256,256]
        layers, in_ch = [], 64
        for idx, out_ch in enumerate(channels):
            stride = 2 if idx in (3, 6) else 1
            layers.append(ConvBnReluSkip(in_ch, out_ch, stride=stride))
            in_ch = out_ch
        self.features = nn.Sequential(*layers)
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(256, num_classes)
        self._init_weights()
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d): nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d): nn.init.constant_(m.weight, 1); nn.init.constant_(m.bias, 0)
    def forward(self, x):
        x = self.stem(x); x = self.features(x)
        return self.fc(torch.flatten(self.avgpool(x), 1))

# ============================================================
# MODEL 3: ResNet WITHOUT Skip Connections (PlainBlock)
# ============================================================
class PlainBlock(nn.Module):
    '''Two-conv block WITHOUT skip connection: y = F(x)'''
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU(inplace=True)
    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.relu(self.bn2(self.conv2(out)))  # ❌ No + x
        return out

class ResNetNoSkip(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.initial = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1, bias=False), nn.BatchNorm2d(64), nn.ReLU(inplace=True))
        self.layer1 = nn.Sequential(PlainBlock(64,64), PlainBlock(64,64), PlainBlock(64,64))
        self.layer2 = nn.Sequential(PlainBlock(64,128,stride=2), PlainBlock(128,128), PlainBlock(128,128))
        self.layer3 = nn.Sequential(PlainBlock(128,256,stride=2), PlainBlock(256,256), PlainBlock(256,256))
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(256, num_classes)
        self._init_weights()
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d): nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d): nn.init.constant_(m.weight, 1); nn.init.constant_(m.bias, 0)
    def forward(self, x):
        x = self.initial(x); x = self.layer1(x); x = self.layer2(x); x = self.layer3(x)
        return self.fc(torch.flatten(self.avgpool(x), 1))

# ============================================================
# MODEL 4: ResNet WITH Skip Connections (BasicBlock) ✅
# ============================================================
class BasicBlock(nn.Module):
    '''Two-conv block WITH skip connection: y = F(x) + x'''
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU(inplace=True)
        self.skip = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.skip = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False), nn.BatchNorm2d(out_ch))
    def forward(self, x):
        identity = self.skip(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += identity  # ✅ F(x) + x
        return self.relu(out)

class ResNetWithSkip(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.initial = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1, bias=False), nn.BatchNorm2d(64), nn.ReLU(inplace=True))
        self.layer1 = nn.Sequential(BasicBlock(64,64), BasicBlock(64,64), BasicBlock(64,64))
        self.layer2 = nn.Sequential(BasicBlock(64,128,stride=2), BasicBlock(128,128), BasicBlock(128,128))
        self.layer3 = nn.Sequential(BasicBlock(128,256,stride=2), BasicBlock(256,256), BasicBlock(256,256))
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(256, num_classes)
        self._init_weights()
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d): nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d): nn.init.constant_(m.weight, 1); nn.init.constant_(m.bias, 0)
    def forward(self, x):
        x = self.initial(x); x = self.layer1(x); x = self.layer2(x); x = self.layer3(x)
        return self.fc(torch.flatten(self.avgpool(x), 1))

print('All 4 models defined successfully!')
for name, cls in [('SimpleCNN', SimpleCNN), ('SimpleCNNSkip', SimpleCNNSkip),
                  ('ResNetNoSkip', ResNetNoSkip), ('ResNetWithSkip', ResNetWithSkip)]:
    m = cls()
    p = sum(p.numel() for p in m.parameters())
    print(f'  {name:20s} — {p:,} parameters')

### 11.2 Training Infrastructure

We train all 4 models with identical hyperparameters and track:
- **Gradient norm (L2)** for every conv layer at every epoch
- **Weight delta (L2)** — how much each layer's weights changed
- **Weight norm (L2)** — total weight magnitude
- **Train/Test accuracy and loss**

This gives us **per-layer evidence** of whether backpropagation can update weights.

In [ ]:
# ============================================================
# TRAINING FUNCTION — Tracks per-layer gradients for every conv
# ============================================================
def get_all_conv_layers(model):
    '''Extract ALL conv layers with their full path names.'''
    conv_layers = {}
    for name, module in model.named_modules():
        if isinstance(module, nn.Conv2d):
            conv_layers[name] = module
    return conv_layers

def train_model_with_tracking(model, model_name, epochs=EPOCHS):
    '''Train a model and track per-layer gradients, weight norms, deltas.'''
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=LR)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=max(1, epochs//2), gamma=0.5)
    criterion = nn.CrossEntropyLoss()
    conv_layers = get_all_conv_layers(model)
    
    history = {
        'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': [],
        'epoch_grad_norms': {n: [] for n in conv_layers},
        'weight_norms':     {n: [] for n in conv_layers},
        'weight_deltas':    {n: [] for n in conv_layers},
    }
    prev_weights = {n: layer.weight.data.clone() for n, layer in conv_layers.items()}
    
    print(f'\n{"="*60}')
    print(f'Training: {model_name} ({len(conv_layers)} conv layers tracked)')
    print(f'{"="*60}')
    
    for epoch in range(epochs):
        # --- TRAIN ---
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        batch_grads = defaultdict(list)
        
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            
            # Track per-batch gradients for every conv layer
            for n, layer in conv_layers.items():
                if layer.weight.grad is not None:
                    batch_grads[n].append(layer.weight.grad.data.norm(2).item())
            
            optimizer.step()
            running_loss += loss.item()
            _, pred = model(images).max(1) if False else (None, loss)  # dummy
            with torch.no_grad():
                out = model(images)
                _, pred = out.max(1)
                total += labels.size(0)
                correct += pred.eq(labels).sum().item()
        
        # Epoch-level gradient norms (mean of batch norms)
        for n in conv_layers:
            norms = batch_grads[n]
            history['epoch_grad_norms'][n].append(np.mean(norms) if norms else 0.0)
        
        # Weight norms & deltas
        for n, layer in conv_layers.items():
            history['weight_norms'][n].append(layer.weight.data.norm(2).item())
            history['weight_deltas'][n].append((layer.weight.data - prev_weights[n]).norm(2).item())
            prev_weights[n] = layer.weight.data.clone()
        
        train_loss = running_loss / len(train_loader)
        train_acc = 100.0 * correct / total
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        
        # --- EVAL ---
        model.eval()
        running_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                out = model(images)
                running_loss += criterion(out, labels).item()
                _, pred = out.max(1)
                total += labels.size(0)
                correct += pred.eq(labels).sum().item()
        
        test_loss = running_loss / len(test_loader)
        test_acc = 100.0 * correct / total
        history['test_loss'].append(test_loss)
        history['test_acc'].append(test_acc)
        scheduler.step()
        
        # Print summary
        layer_names = list(conv_layers.keys())
        first_grad = history['epoch_grad_norms'][layer_names[0]][-1]
        last_grad  = history['epoch_grad_norms'][layer_names[-1]][-1]
        ratio = first_grad / (last_grad + 1e-10)
        print(f'  Epoch {epoch+1:2d}/{epochs} | '
              f'TrainAcc: {train_acc:.1f}% | TestAcc: {test_acc:.1f}% | '
              f'GradRatio(first/last): {ratio:.4f}')
    
    print(f'  Done! Best Test Acc: {max(history["test_acc"]):.2f}%')
    return history, conv_layers

print('Training infrastructure ready.')

### 11.3 Train All Four Models

⏱️ **Estimated time**: ~20-30 minutes on Colab T4 GPU (10 epochs × 4 models)

In [ ]:
# ============================================================
# TRAIN ALL 4 MODELS
# ============================================================
models_config = [
    ('Simple CNN\n(No Skip)', SimpleCNN()),
    ('CNN + Skip', SimpleCNNSkip()),
    ('ResNet\n(No Skip)', ResNetNoSkip()),
    ('ResNet\n(With Skip) ✅', ResNetWithSkip()),
]

all_results = {}  # model_name -> (history, conv_layers)
start_time = time.time()

for model_name, model in models_config:
    history, conv_layers = train_model_with_tracking(model, model_name)
    all_results[model_name] = (history, list(conv_layers.keys()))

total_time = time.time() - start_time
print(f'\n{"="*60}')
print(f'ALL TRAINING COMPLETE in {total_time/60:.1f} minutes')
print(f'{"="*60}')

---
## 12. Comparison Graphs — Proving Vanishing Gradient Removal

### Why Per-Layer Analysis Matters

Top-line accuracy can be misleading. Two models with similar accuracy can have
**very different internal gradient dynamics**. The real proof lies in:

1. **Per-layer gradient norms**: Are early layers receiving gradients?
2. **Per-layer weight deltas**: Are early layers actually updating?
3. **Gradient ratio (first/last)**: Is gradient flow balanced?

### Identity Mapping: The Core Issue

| Forward Pass | Backward Pass (Gradient) | Problem |
|-------------|-------------------------|----------|
| $y = \\mathcal{F}(x)$ (no skip) | $\\frac{\\partial y}{\\partial x} = \\frac{\\partial \\mathcal{F}}{\\partial x}$ | Can vanish if $\\frac{\\partial \\mathcal{F}}{\\partial x} \\to 0$ |
| $y = \\mathcal{F}(x) + x$ (skip) | $\\frac{\\partial y}{\\partial x} = \\frac{\\partial \\mathcal{F}}{\\partial x} + \\mathbf{I}$ | **Always ≥ I** — gradient highway! |

Without `+ x`: if any layer's Jacobian is small, **all upstream layers starve**.
With `+ x`: the identity term **guarantees** gradient flows through, regardless of $\\mathcal{F}$.

In [ ]:
# ============================================================
# PLOT 1: Per-Layer Gradient Norms at Final Epoch (All 4 Models)
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(18, 14))
fig.suptitle('Per-Layer Gradient Norms at Final Epoch — All 4 Models\n'
             '(Lower bars in early layers = vanishing gradients = weights NOT updating)',
             fontsize=15, fontweight='bold', y=1.02)

colors_map = {'early': '#e74c3c', 'mid': '#f39c12', 'late': '#27ae60'}

for idx, (model_name, (history, layer_names)) in enumerate(all_results.items()):
    ax = axes[idx // 2][idx % 2]
    final_grads = [history['epoch_grad_norms'][n][-1] for n in layer_names]
    n_layers = len(layer_names)
    
    # Color by depth: early=red, mid=orange, late=green
    colors = []
    for i in range(n_layers):
        if i < n_layers // 3: colors.append('#e74c3c')  # early
        elif i < 2 * n_layers // 3: colors.append('#f39c12')  # mid
        else: colors.append('#27ae60')  # late
    
    short_names = [f'L{i+1}' for i in range(n_layers)]
    bars = ax.bar(short_names, final_grads, color=colors, edgecolor='black', linewidth=0.5)
    ax.set_title(model_name, fontsize=13, fontweight='bold')
    ax.set_ylabel('Gradient Norm (L2)', fontsize=11)
    ax.set_xlabel('Layer Index (L1=earliest → last=deepest)', fontsize=10)
    ax.set_yscale('log')
    ax.grid(True, alpha=0.3, axis='y')
    ax.tick_params(axis='x', rotation=45, labelsize=7)
    
    # Annotate ratio
    ratio = final_grads[0] / (final_grads[-1] + 1e-10)
    ax.text(0.95, 0.95, f'Ratio(L1/Last): {ratio:.4f}',
            transform=ax.transAxes, fontsize=10, ha='right', va='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.tight_layout()
plt.savefig('comparison_gradient_norms_per_layer.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: comparison_gradient_norms_per_layer.png')

**🔍 What to observe**: In no-skip models (top-left, bottom-left), early layer bars (red) are
**orders of magnitude shorter** than late layer bars (green). This means early layers receive
almost no gradient → their weights **cannot update** during backpropagation.

In skip models (top-right, bottom-right), bars are much more **uniform** across all layers.

In [ ]:
# ============================================================
# PLOT 2: Per-Layer Weight Deltas at Final Epoch
# (How much did each layer's weights ACTUALLY change?)
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(18, 14))
fig.suptitle('Per-Layer Weight Deltas at Final Epoch — "Can Weights Update?"\n'
             '(Near-zero delta = weight is FROZEN = backprop cannot reach this layer)',
             fontsize=15, fontweight='bold', y=1.02)

for idx, (model_name, (history, layer_names)) in enumerate(all_results.items()):
    ax = axes[idx // 2][idx % 2]
    final_deltas = [history['weight_deltas'][n][-1] for n in layer_names]
    n_layers = len(layer_names)
    
    colors = []
    for i in range(n_layers):
        if i < n_layers // 3: colors.append('#c0392b')
        elif i < 2 * n_layers // 3: colors.append('#e67e22')
        else: colors.append('#2ecc71')
    
    short_names = [f'L{i+1}' for i in range(n_layers)]
    ax.bar(short_names, final_deltas, color=colors, edgecolor='black', linewidth=0.5)
    ax.set_title(model_name, fontsize=13, fontweight='bold')
    ax.set_ylabel('Weight Delta (L2 of Δw)', fontsize=11)
    ax.set_xlabel('Layer Index', fontsize=10)
    ax.set_yscale('log')
    ax.grid(True, alpha=0.3, axis='y')
    ax.tick_params(axis='x', rotation=45, labelsize=7)
    
    # Annotate
    ratio = final_deltas[0] / (final_deltas[-1] + 1e-10)
    ax.text(0.95, 0.95, f'Δw Ratio(L1/Last): {ratio:.4f}',
            transform=ax.transAxes, fontsize=10, ha='right', va='top',
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.savefig('comparison_weight_deltas_per_layer.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: comparison_weight_deltas_per_layer.png')

**🔍 Proof that weights cannot update**: In models without skip connections,
early layers have **near-zero weight deltas** — the optimizer step barely changes them
because the gradient signal has vanished by the time it reaches these layers.

> This is the **direct consequence** of $y = \\mathcal{F}(x)$ without the identity shortcut.
> When $\\frac{\\partial \\mathcal{F}}{\\partial x} \\to 0$ in some layers, the chain rule product collapses.

In [ ]:
# ============================================================
# PLOT 3: Gradient Norm Heatmap — Layers × Epochs (2×2 Grid)
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(20, 14))
fig.suptitle('Gradient Norm Heatmap — Layers × Epochs\n'
             '(Uniform color = healthy flow | Dark early rows = vanishing gradients)',
             fontsize=15, fontweight='bold', y=1.02)

for idx, (model_name, (history, layer_names)) in enumerate(all_results.items()):
    ax = axes[idx // 2][idx % 2]
    data = np.array([history['epoch_grad_norms'][n] for n in layer_names])
    
    im = ax.imshow(data, aspect='auto', cmap='YlOrRd', interpolation='nearest')
    ax.set_yticks(range(0, len(layer_names), max(1, len(layer_names)//10)))
    ax.set_yticklabels([f'L{i+1}' for i in range(0, len(layer_names), max(1, len(layer_names)//10))], fontsize=8)
    ax.set_xlabel('Epoch', fontsize=11)
    ax.set_ylabel('Layer (early → deep)', fontsize=11)
    ax.set_xticks(range(EPOCHS))
    ax.set_xticklabels(range(1, EPOCHS + 1), fontsize=9)
    ax.set_title(model_name, fontsize=13, fontweight='bold')
    plt.colorbar(im, ax=ax, label='Gradient Norm (L2)', shrink=0.8)

plt.tight_layout()
plt.savefig('comparison_gradient_heatmaps.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: comparison_gradient_heatmaps.png')

In [ ]:
# ============================================================
# PLOT 4: Gradient Ratio (First Layer / Last Layer) Over Epochs
# ============================================================
fig, ax = plt.subplots(figsize=(12, 7))

colors = ['#e74c3c', '#3498db', '#e67e22', '#2ecc71']
markers = ['o', 's', '^', 'D']

for (model_name, (history, layer_names)), color, marker in zip(
        all_results.items(), colors, markers):
    first_grads = np.array(history['epoch_grad_norms'][layer_names[0]])
    last_grads  = np.array(history['epoch_grad_norms'][layer_names[-1]])
    ratio = first_grads / (last_grads + 1e-10)
    ax.plot(range(1, EPOCHS+1), ratio, f'-{marker}', color=color,
            label=model_name.replace('\n', ' '), markersize=6, linewidth=2)

ax.axhline(y=1.0, color='black', linestyle='--', linewidth=2, alpha=0.5, label='Ideal (ratio = 1.0)')
ax.set_xlabel('Epoch', fontsize=13)
ax.set_ylabel('Gradient Ratio (First Layer / Last Layer)', fontsize=13)
ax.set_title('Gradient Ratio Over Training — Skip vs No-Skip\n'
             '(Ratio ≈ 1.0 = balanced gradient flow | Ratio ≪ 1 = vanishing gradients)',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=11, loc='best')
ax.grid(True, alpha=0.3)
ax.set_yscale('log')
plt.tight_layout()
plt.savefig('comparison_gradient_ratio.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: comparison_gradient_ratio.png')

In [ ]:
# ============================================================
# PLOT 5: Accuracy Curves — All 4 Models
# ============================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

colors = ['#e74c3c', '#3498db', '#e67e22', '#2ecc71']
for (model_name, (history, _)), color in zip(all_results.items(), colors):
    label = model_name.replace('\n', ' ')
    ax1.plot(range(1, EPOCHS+1), history['train_acc'], '-o', color=color,
            label=label, markersize=4, linewidth=2)
    ax2.plot(range(1, EPOCHS+1), history['test_acc'], '-o', color=color,
            label=label, markersize=4, linewidth=2)

ax1.set_title('Training Accuracy', fontsize=14, fontweight='bold')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy (%)')
ax1.legend(fontsize=10); ax1.grid(True, alpha=0.3)

ax2.set_title('Test Accuracy', fontsize=14, fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)')
ax2.legend(fontsize=10); ax2.grid(True, alpha=0.3)

fig.suptitle('Accuracy Comparison — All 4 Configurations', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('comparison_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: comparison_accuracy.png')

In [ ]:
# ============================================================
# PLOT 6: Loss Curves — All 4 Models
# ============================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

colors = ['#e74c3c', '#3498db', '#e67e22', '#2ecc71']
for (model_name, (history, _)), color in zip(all_results.items(), colors):
    label = model_name.replace('\n', ' ')
    ax1.plot(range(1, EPOCHS+1), history['train_loss'], '-o', color=color,
            label=label, markersize=4, linewidth=2)
    ax2.plot(range(1, EPOCHS+1), history['test_loss'], '-o', color=color,
            label=label, markersize=4, linewidth=2)

ax1.set_title('Training Loss', fontsize=14, fontweight='bold')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.legend(fontsize=10); ax1.grid(True, alpha=0.3)

ax2.set_title('Test Loss', fontsize=14, fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
ax2.legend(fontsize=10); ax2.grid(True, alpha=0.3)

fig.suptitle('Loss Comparison — All 4 Configurations', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('comparison_loss.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: comparison_loss.png')

In [ ]:
# ============================================================
# PLOT 7: Final Gradient Ratio (Bar Chart) — One Bar Per Model
# ============================================================
fig, ax = plt.subplots(figsize=(10, 6))

model_names_short = []
final_ratios = []
colors = ['#e74c3c', '#3498db', '#e67e22', '#2ecc71']

for model_name, (history, layer_names) in all_results.items():
    first_grad = history['epoch_grad_norms'][layer_names[0]][-1]
    last_grad  = history['epoch_grad_norms'][layer_names[-1]][-1]
    ratio = first_grad / (last_grad + 1e-10)
    model_names_short.append(model_name.replace('\n', ' '))
    final_ratios.append(ratio)

bars = ax.bar(model_names_short, final_ratios, color=colors, edgecolor='black', linewidth=1.5)
ax.axhline(y=1.0, color='black', linestyle='--', linewidth=2, alpha=0.5, label='Ideal = 1.0')

# Annotate bars
for bar, ratio in zip(bars, final_ratios):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{ratio:.4f}', ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_ylabel('Gradient Ratio (First / Last Layer)', fontsize=13)
ax.set_title('Final Epoch: Gradient Ratio per Model\n'
             '(Closer to 1.0 = better gradient flow = no vanishing gradient)',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('comparison_final_gradient_ratio.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: comparison_final_gradient_ratio.png')

---
## 13. Summary Comparison Table (Computed from Training)


In [ ]:
# ============================================================
# SUMMARY TABLE — Computed from actual training results
# ============================================================
print('\n' + '='*100)
print(f'{"Model":<25} {"Best Test Acc":>13} {"Final Train Loss":>16} {"Grad Ratio":>12} '
      f'{"L1 Grad Norm":>14} {"Last Grad Norm":>15} {"L1 Wt Delta":>13} {"Last Wt Delta":>14}')
print('='*100)

for model_name, (history, layer_names) in all_results.items():
    best_acc = max(history['test_acc'])
    final_train_loss = history['train_loss'][-1]
    
    first_grad = history['epoch_grad_norms'][layer_names[0]][-1]
    last_grad  = history['epoch_grad_norms'][layer_names[-1]][-1]
    ratio = first_grad / (last_grad + 1e-10)
    
    first_delta = history['weight_deltas'][layer_names[0]][-1]
    last_delta  = history['weight_deltas'][layer_names[-1]][-1]
    
    name = model_name.replace('\n', ' ')
    print(f'{name:<25} {best_acc:>12.2f}% {final_train_loss:>16.4f} {ratio:>12.6f} '
          f'{first_grad:>14.6f} {last_grad:>15.6f} {first_delta:>13.6f} {last_delta:>14.6f}')

print('='*100)
print()
print('KEY OBSERVATIONS:')
print('  → Models WITHOUT skip connections have gradient ratio ≪ 1 (vanishing gradients)')
print('  → Models WITH skip connections have gradient ratio ≈ 1 (healthy gradient flow)')
print('  → Early layer weight deltas are much larger with skip connections (weights CAN update)')
print('  → ResNet + Skip achieves the best accuracy due to effective learning at ALL depths')

---
## 14. Per-Layer Proof: Identity Mapping vs Residual Mapping

### The Backpropagation Failure Without Skip Connections

Consider a network with $L$ layers. During backpropagation, the gradient at layer $l$ is:

$$\\frac{\\partial \\mathcal{L}}{\\partial W_l} = \\frac{\\partial \\mathcal{L}}{\\partial a_L} \\cdot \\underbrace{\\prod_{i=l}^{L-1} \\frac{\\partial a_{i+1}}{\\partial a_i}}_{\\text{chain rule product}}$$

#### Without Skip ($y = F(x)$): Identity Mapping Fails

Each factor in the product is $\\frac{\\partial \\mathcal{F}_i}{\\partial x_i}$. If even a few of these are $< 1$:

| Layer | Gradient Factor | Cumulative Product |
|-------|:-:|:-:|
| Layer 20 (output) | 1.0 | 1.0 |
| Layer 15 | 0.8 | 0.8 |
| Layer 10 | 0.5 | 0.4 |
| Layer 5 | 0.3 | 0.12 |
| Layer 1 (input) | 0.2 | **0.024** ← almost zero! |

**Result**: Layer 1 receives only 2.4% of the gradient signal. Its weights **barely change**.

#### With Skip ($y = F(x) + x$): Addition Saves the Day

Now each factor becomes $\\frac{\\partial \\mathcal{F}_i}{\\partial x_i} + \\mathbf{I}$. Even if $\\frac{\\partial \\mathcal{F}_i}{\\partial x_i}$ is small:

| Layer | Gradient Factor | Cumulative Product |
|-------|:-:|:-:|
| Layer 20 (output) | 1.0 + 1.0 = 2.0 | 2.0 |
| Layer 15 | 0.8 + 1.0 = 1.8 | 3.6 |
| Layer 10 | 0.5 + 1.0 = 1.5 | 5.4 |
| Layer 5 | 0.3 + 1.0 = 1.3 | 7.0 |
| Layer 1 (input) | 0.2 + 1.0 = 1.2 | **8.4** ← strong signal! |

**Result**: Layer 1 receives a **strong gradient signal**. Weights update effectively.

> **The `+ x` in $y = F(x) + x$ is the single most important innovation in ResNet.**
> It transforms the gradient from a vanishing product to a growing sum.

In [ ]:
# ============================================================
# PLOT 8: Per-Layer Gradient Flow Comparison
# (Overlaid line plots for all 4 models on same axes)
# ============================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

colors = ['#e74c3c', '#3498db', '#e67e22', '#2ecc71']

# Left: Gradient norms across layers (final epoch)
for (model_name, (history, layer_names)), color in zip(all_results.items(), colors):
    final_grads = [history['epoch_grad_norms'][n][-1] for n in layer_names]
    # Normalize layer indices to [0, 1] for fair comparison (different total layers)
    x = np.linspace(0, 1, len(final_grads))
    ax1.plot(x, final_grads, '-o', color=color, label=model_name.replace('\n', ' '),
            markersize=4, linewidth=2, alpha=0.8)

ax1.set_xlabel('Relative Depth (0 = first layer → 1 = last layer)', fontsize=12)
ax1.set_ylabel('Gradient Norm (L2) — log scale', fontsize=12)
ax1.set_title('Gradient Flow Across Network Depth\n'
              '(Flat line = healthy | Declining = vanishing)', fontsize=13, fontweight='bold')
ax1.set_yscale('log')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Right: Weight deltas across layers (final epoch)
for (model_name, (history, layer_names)), color in zip(all_results.items(), colors):
    final_deltas = [history['weight_deltas'][n][-1] for n in layer_names]
    x = np.linspace(0, 1, len(final_deltas))
    ax2.plot(x, final_deltas, '-s', color=color, label=model_name.replace('\n', ' '),
            markersize=4, linewidth=2, alpha=0.8)

ax2.set_xlabel('Relative Depth (0 = first layer → 1 = last layer)', fontsize=12)
ax2.set_ylabel('Weight Delta (L2 of Δw) — log scale', fontsize=12)
ax2.set_title('Weight Update Magnitude Across Depth\n'
              '(Flat = all layers update | Declining = early layers frozen)', fontsize=13, fontweight='bold')
ax2.set_yscale('log')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('comparison_gradient_flow_across_depth.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: comparison_gradient_flow_across_depth.png')

---
## 15. Final Verdict

### Evidence Summary

| Evidence | No-Skip Models | Skip Models |
|----------|:-:|:-:|
| Early layer gradient norms | 📉 Orders of magnitude weaker | ✅ Comparable to later layers |
| Early layer weight deltas | 📉 Near-zero (frozen) | ✅ Meaningful updates |
| Gradient ratio (first/last) | ≪ 1.0 ❌ | ≈ 1.0 ✅ |
| Gradient heatmap | Dark early rows (starving) | Uniform color (healthy) |
| Test accuracy | Lower | Higher |

### Mathematical Guarantee

The addition $y = \\mathcal{F}(x) + x$ adds $+\\mathbf{I}$ to the Jacobian at every layer:

$$\\frac{\\partial y}{\\partial x} = \\frac{\\partial \\mathcal{F}(x)}{\\partial x} + \\mathbf{I} \\geq \\mathbf{I}$$

This means **no matter how deep the network**, the gradient at any layer is at least $\\mathbf{I}$,
creating a **gradient highway** that prevents vanishing.

### Conclusion

> 🏆 **Skip connections are the single most important architectural innovation in ResNet.**
> They provide a mathematical guarantee that gradients flow to all layers,
> enabling effective training of networks with 100+ layers.
> The per-layer evidence above **proves** this claim with real training data.